# Test rapide de HedgeNet (S2)

Objectif : vérifier que le réseau tourne et que la variance du P&L baisse.
Utilise le simulateur et le P&L d'Aymane (`src/data.py`, `src/pricing.py`).

In [ ]:
import sys
sys.path.append("..")  # pour importer le dossier src/ depuis notebooks/

import torch
import matplotlib.pyplot as plt

from src.data import simulate_gbm
from src.pricing import bs_call_price, bs_delta_strategy, naive_strategy, compute_hedging_pnl, summarize_pnl
from src.hedge_net import hedgenet_strategy, train_hedgenet

## 1. Paramètres du marché

In [ ]:
S0, K, r, sigma, T = 100.0, 100.0, 0.03, 0.2, 1.0
n_steps = 30
c = 1e-3  # coût de transaction proportionnel

## 2. Petites données pour le test (le vrai entraînement sur 100 000 trajectoires est prévu en S3)

In [ ]:
S_train = simulate_gbm(S0, r, sigma, T, n_steps, n_paths=10_000, seed=42)
S_test = simulate_gbm(S0, r, sigma, T, n_steps, n_paths=5_000, seed=123)
print(S_train.shape, S_test.shape)

## 3. Entraînement rapide

In [ ]:
model, losses = train_hedgenet(S_train, K, T, c=c, epochs=10)

In [ ]:
plt.plot(range(1, len(losses) + 1), losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Var(P&L)")
plt.title("Entraînement de HedgeNet")
plt.show()

## 4. Première comparaison sur le test (indicative)

In [ ]:
premium = bs_call_price(S0, K, T, r, sigma)  # prime reçue en vendant le Call

with torch.no_grad():
    strategies = {
        "Naïve (sans couverture)": naive_strategy(S_test),
        "Delta Black-Scholes": bs_delta_strategy(S_test, K, T, r, sigma),
        "HedgeNet": hedgenet_strategy(model, S_test, K, T),
    }

    for name, deltas in strategies.items():
        pnl, costs = compute_hedging_pnl(S_test, deltas, K, c, include_premium=True, premium=premium)
        print(name, summarize_pnl(pnl, costs))